# 01: Screen, Build, and Annotate the 260-Image Benchmark

This notebook first removes semantic dataset failures such as collages, heavy overlays, class mismatches, and unrelated multi-dish scenes. Rejected candidates are replaced within the same source class. Only then are 260 accepted images assigned to the sealed 156/52/52 train, validation, and test split.

Annotator A labels all 260 images. Annotator B independently labels the 104 validation and test images, giving the final evaluation set double-annotated and adjudicated ground truth.

Primary task: **per-image visible-component recognition**. Recipe knowledge, hidden ingredients, web associations, food naming, and nutrition are not ground truth for this task.

## Define the target and frozen ontology

The ontology uses canonical multiword component labels and gives every label a visual-evidence rule. Recipe-writing fragments and preparation terms such as `all`, `at`, `freshly`, and `coarsely` are not model targets.

A valid mixed dish on one plate remains eligible. A collage, graphic montage, wrong-class image, or scene with no primary dish does not. Quality screening happens before split membership exists, while ingredient annotation remains blinded to source class and split.

In [ ]:
# Kaggle setup: clone the project when the notebook was imported without repository files.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/ThatTryHard/indonesian-food-vlm-analyzer.git"
REPOSITORY_REVISION = os.environ.get("FOOD_VLM_REVISION", "main")
PROJECT_ROOT = Path(os.environ.get("FOOD_VLM_PROJECT_ROOT", "/kaggle/working/indonesian-food-vlm-analyzer"))

if not (PROJECT_ROOT / "src").exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPOSITORY_REVISION,
        REPOSITORY_URL, str(PROJECT_ROOT),
    ], check=True)

# Notebook 01 only needs packages already supplied by Kaggle's pinned base image.
# Replacing NumPy/Pandas in a live notebook can mix old and new binary modules.
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Environment: Kaggle pinned base packages (no in-kernel replacement)")
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import hashlib
import json
import shutil

import kagglehub
import pandas as pd
from IPython.display import display

from src.artifacts import project_protocol_digest
from src.config import artifact_dir, load_config
from src.data import manifest_digest, sha256_file
from src.ontology import IngredientOntology
from src.vlm import build_visible_prompt

config_path = PROJECT_ROOT / "configs/project.json"
ontology_path = PROJECT_ROOT / "data/ontology/visible_ingredients.json"
config = load_config(config_path)
ontology = IngredientOntology.from_json(ontology_path)
ARTIFACT_ROOT = artifact_dir(config)
BENCHMARK_DIR = ARTIFACT_ROOT / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

# Resume a partial quality screen or annotation pass from a previously exported packet.
packet_override = os.environ.get("FOOD_VLM_BENCHMARK_PACKET")
packet_source = Path(packet_override) if packet_override else None
if packet_source is None and Path("/kaggle/input").exists():
    attached_manifests = list(Path("/kaggle/input").rglob("benchmark_manifest.csv"))
    attached_archives = list(Path("/kaggle/input").rglob("benchmark_packet.zip"))
    candidates = attached_manifests + attached_archives
    if len(candidates) == 1:
        packet_source = candidates[0]
    elif len(candidates) > 1:
        print("Multiple prior packets found; set FOOD_VLM_BENCHMARK_PACKET explicitly.")
local_packet_exists = any(
    (BENCHMARK_DIR / filename).exists()
    for filename in ["quality_screen.csv", "benchmark_manifest.csv"]
)
if packet_source is not None and not local_packet_exists:
    if not packet_source.exists():
        raise FileNotFoundError(packet_source)
    if packet_source.is_file() and packet_source.suffix.lower() == ".zip":
        shutil.unpack_archive(str(packet_source), str(BENCHMARK_DIR))
    else:
        source_dir = packet_source.parent if packet_source.is_file() else packet_source
        packet_filenames = {
            "quality_candidate_pool.csv", "quality_screen.csv", "benchmark_manifest.csv",
            "manifest_lock.json", "image_inventory.csv", "corrupt_images.csv",
            "annotations_annotator_a.csv", "annotations_annotator_b.csv",
        }
        for filename in packet_filenames:
            source_file = source_dir / filename
            if source_file.is_file():
                shutil.copy2(source_file, BENCHMARK_DIR / filename)
    print("Restored writable benchmark packet from:", packet_source)

ontology_table = pd.DataFrame([
    {"id": label.id, "category": label.category, "visual_rule": label.hint}
    for label in ontology.labels
])
print("Primary task:", config["project"]["primary_task"])
print("Ontology version:", ontology.version, "| labels:", len(ontology.ids))
display(ontology_table)

In [ ]:
# Resolve the exact Kaggle dataset by its frozen slug.
dataset_config = config["datasets"]["indonesian_target"]
explicit_target = os.environ.get(dataset_config["path_env"])
TARGET_DATASET_ROOT = Path(explicit_target) if explicit_target else Path(kagglehub.dataset_download(dataset_config["slug"]))

print("Dataset slug:", dataset_config["slug"])
print("Resolved root:", TARGET_DATASET_ROOT)
if not TARGET_DATASET_ROOT.exists():
    raise FileNotFoundError(TARGET_DATASET_ROOT)

## Stage 1: Screen image quality before sealing

File decoding and perceptual hashes cannot detect a collage or a wrong-class photograph. The interface therefore reviews deterministic reserve candidates class by class. Choose **Accept** for one assessable food photograph. Otherwise choose the exact rejection reason. After a rejection, the next reserve candidate from that same class appears automatically.

The source class is visible only during this quality check so a class mismatch can be detected. It is hidden during ingredient annotation.

In [ ]:
candidate_pool_path = BENCHMARK_DIR / "quality_candidate_pool.csv"
quality_screen_path = BENCHMARK_DIR / "quality_screen.csv"
manifest_path = BENCHMARK_DIR / "benchmark_manifest.csv"

if manifest_path.exists() and (not candidate_pool_path.exists() or not quality_screen_path.exists()):
    raise RuntimeError(
        "An older pre-quality-screen packet was detected. Start a fresh Notebook 01 session; "
        "do not reuse that manifest."
    )
if not candidate_pool_path.exists() and not manifest_path.exists():
    subprocess.run([
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_benchmark.py"),
        "prepare",
        "--dataset-root", str(TARGET_DATASET_ROOT),
        "--output-dir", str(BENCHMARK_DIR),
    ], check=True)

candidate_pool = pd.read_csv(candidate_pool_path, keep_default_na=False)
quality_screen = pd.read_csv(quality_screen_path, keep_default_na=False)
from src.quality import quality_screen_progress, validate_quality_screen
quality_screen = validate_quality_screen(
    quality_screen, candidate_pool, config["benchmark"]["samples_per_class"], require_complete=False
)
print(json.dumps(
    quality_screen_progress(quality_screen, candidate_pool, config["benchmark"]["samples_per_class"]),
    indent=2, sort_keys=True,
))

In [ ]:
# Review until every class has 20 accepted images. Each click is saved to quality_screen.csv.
from src.quality_ui import QualityScreenApp

current_quality_progress = quality_screen_progress(
    quality_screen, candidate_pool, config["benchmark"]["samples_per_class"]
)
if not manifest_path.exists() and not current_quality_progress["complete"]:
    quality_app = QualityScreenApp(
        screen=quality_screen,
        candidate_pool=candidate_pool,
        image_root=TARGET_DATASET_ROOT,
        output_csv=quality_screen_path,
        samples_per_class=config["benchmark"]["samples_per_class"],
    )
    quality_app.display()
elif current_quality_progress["complete"] and not manifest_path.exists():
    print("Quality screen is complete. Run the sealing/export cell below.")
else:
    print("Quality screen is already complete and the benchmark is sealed.")

## Finalize the quality screen and seal the benchmark

Rerun the next cell after screening. If the screen is incomplete, it exports a resumable packet and leaves the manifest unsealed. When every class has 20 accepted images, it assigns the 12/4/4 split, creates the 260-row primary sheet and 104-row secondary sheet, and cryptographically locks the result.

In [ ]:
quality_screen = pd.read_csv(quality_screen_path, keep_default_na=False)
quality_progress = quality_screen_progress(
    quality_screen, candidate_pool, config["benchmark"]["samples_per_class"]
)
print(json.dumps(quality_progress, indent=2, sort_keys=True))

if quality_progress["complete"] and not manifest_path.exists():
    subprocess.run([
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_benchmark.py"),
        "seal",
        "--output-dir", str(BENCHMARK_DIR),
    ], check=True)

archive_base = ARTIFACT_ROOT / "benchmark_packet"
archive_path = shutil.make_archive(str(archive_base), "zip", BENCHMARK_DIR)
print("Download this resumable checkpoint:", archive_path)

BENCHMARK_SEALED = manifest_path.exists()
if BENCHMARK_SEALED:
    manifest = pd.read_csv(manifest_path, keep_default_na=False)
    lock = json.loads((BENCHMARK_DIR / "manifest_lock.json").read_text(encoding="utf-8"))
    assert len(manifest) == 260 and manifest["sample_id"].is_unique
    assert manifest.groupby("food_class").size().eq(20).all()
    assert manifest_digest(manifest) == lock["manifest_sha256"]
    assert sha256_file(candidate_pool_path) == lock["candidate_pool_sha256"]
    assert sha256_file(quality_screen_path) == lock["quality_screen_sha256"]
    assert lock["screened_before_split"] is True
    assert lock["annotation_rows"] == {"annotator_a": 260, "annotator_b": 104}
    assert sha256_file(ontology_path) == lock["ontology_sha256"]
    assert sha256_file(config_path) == lock["config_sha256"]
    assert hashlib.sha256(build_visible_prompt(ontology).encode("utf-8")).hexdigest() == lock["vlm_prompt_sha256"]
    assert project_protocol_digest(PROJECT_ROOT) == lock["project_protocol_sha256"]
    per_class_splits = manifest.groupby(["food_class", "split"]).size().unstack(fill_value=0)
    assert per_class_splits["train"].eq(12).all()
    assert per_class_splits["validation"].eq(4).all()
    assert per_class_splits["test"].eq(4).all()
    print("Benchmark sealed after semantic quality screening.")
    display(per_class_splits)
else:
    print("Benchmark is not sealed yet. Continue the quality interface, then rerun this cell.")

## Stage 2: Annotate visible ingredients

Read `docs/ANNOTATION_GUIDE.md` before starting. Annotator A labels all 260 images. Annotator B works independently on the 104 validation and test images only. The annotation interface hides source class and split. Do not inspect the other person's sheet.

In [ ]:
print((PROJECT_ROOT / "docs/ANNOTATION_GUIDE.md").read_text(encoding="utf-8"))

In [ ]:
# Choose exactly one independent pass. Use a separate session for annotator_b.
ANNOTATOR_ID = "annotator_a"
ANNOTATION_READY = bool(BENCHMARK_SEALED)

if ANNOTATOR_ID not in {"annotator_a", "annotator_b"}:
    raise ValueError("ANNOTATOR_ID must be annotator_a or annotator_b")
if ANNOTATION_READY:
    annotation_path = BENCHMARK_DIR / f"annotations_{ANNOTATOR_ID}.csv"
    annotation_manifest = (
        manifest
        if ANNOTATOR_ID == "annotator_a"
        else manifest[manifest["split"].isin(config["benchmark"]["secondary_annotation_splits"])].copy()
    )
    sheet = pd.read_csv(annotation_path, keep_default_na=False)
    from src.annotations import validate_annotation_sheet
    sheet = validate_annotation_sheet(sheet, annotation_manifest, ontology, require_complete=False)
    expected_rows = 260 if ANNOTATOR_ID == "annotator_a" else 104
    assert len(sheet) == expected_rows
    print(f"Editing {ANNOTATOR_ID}: {len(sheet)} images")
    print("Do not open the other annotator's CSV during this pass.")
else:
    print("Annotation is blocked until quality screening is complete and the manifest is sealed.")

In [ ]:
from src.annotation_ui import AnnotationApp

if ANNOTATION_READY:
    app = AnnotationApp(
        sheet=sheet,
        ontology=ontology,
        image_root=TARGET_DATASET_ROOT,
        output_csv=annotation_path,
    )
    app.display()
else:
    print("Return to the quality-screen interface above.")

## Annotation progress and export

Rerun this cell whenever you stop. The interface saves each confirmed image immediately, while this cell packages the full quality screen, lock, manifest, and annotation sheets for safe resumption.

In [ ]:
from src.annotations import annotation_progress, validate_annotation_sheet

if ANNOTATION_READY:
    saved_sheet = pd.read_csv(annotation_path, keep_default_na=False)
    progress = annotation_progress(saved_sheet)
    print(progress)
    if progress["remaining"] == 0:
        validate_annotation_sheet(saved_sheet, annotation_manifest, ontology, require_complete=True)
        print(f"{ANNOTATOR_ID} is structurally complete: {len(saved_sheet)} images.")
    else:
        print("Resume this annotation pass before Notebook 02.")

archive_path = shutil.make_archive(str(ARTIFACT_ROOT / "benchmark_packet"), "zip", BENCHMARK_DIR)
print("Download and preserve:", archive_path)

## Handoff to Notebook 02

Proceed only when Annotator A has completed 260 images and Annotator B has independently completed the 104 validation/test images. Notebook 02 measures agreement on those 104 images, adjudicates their disagreements, combines them with the primary training labels, and only then trains and evaluates models.